In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, RrfQuery, Rrf, Document
import openai

In [2]:
COLLECTION_NAME="cm_interventions_hybrid"

In [3]:
def embed_text(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [4]:
qdrant_host = "http://localhost:6333"
qdrant_client = QdrantClient(qdrant_host)

In [5]:
def retrieve(query, top_k=5):
    vector = embed_text(query)
    
    hits = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=vector,
                using="text-embedding-3-small",
                limit=top_k // 2
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=top_k // 2
            )
        ],
        query=RrfQuery(rrf=Rrf(weights=[1, 1])),
        limit=top_k
    ).points


    return hits

In [6]:
query = "roller issue"
results = retrieve(query, top_k=20)

In [7]:
results

[ScoredPoint(id='c667df92-92c8-4822-8236-acfa4425d340', version=5, score=0.5, payload={'id': 'INT-2022-0518', 'date_start': '2022-07-03 07:42', 'machine': 'CR-100', 'duration_min': 273, 'summary': '[FAULT_CODE] R-008\n[RELATED_INTERVENTION] INT-2022-0450\n[EVENT] Emergency stop activated on rolling mill. Strip broke at entry pinch roll — operator E-Stop.\n[COMMENTS] Fault R-008: strip breakage at entry, operator-triggered E-Stop. Strip removed from mill. Roll gap and tension settings reviewed. Root cause: coupling misalignment following recent maintenance work. Inspection interval reduced from quarterly to monthly for this subsystem.', 'embedding_model': 'text-embedding-3-small', 'created_at': '2026-04-06T10:07:51.838215'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id='df73d627-4148-461e-a8c1-1c551ddc7245', version=10, score=0.5, payload={'id': 'INT-2023-0139', 'date_start': '2023-02-09 07:53', 'machine': 'CB-200', 'duration_min': 295, 'summary': '[FAULT_CODE] B-006\

## Reranking with Cohere

In [8]:
import cohere

In [9]:
cohere_client = cohere.ClientV2()


In [10]:
to_rerank = [hit.payload["summary"] for hit in results]
to_rerank[:2]

In [11]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

